# 21_operations: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/21_operations/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Bound retries and compact versioned rows.
Run: uv run python modules/21_operations/examples/01_from_scratch.py
CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
Read the module's setup for optional extras and server requirements.
"""

from common.ops.ingestion import ingest, stable_id
from tinyvec.storage import VersionedRows


def main():
    store = VersionedRows()

    def write(batch):
        for key, value in batch:
            store.upsert(key, value)

    rows = [(stable_id("seller-a", str(i)), {"price": i}) for i in range(100)]
    first = ingest(rows, write, batch_size=16, pending_limit=2)
    ingest(rows, write, batch_size=16)
    assert len(store.live()) == 100 and len(store.rows) == 200
    for key, _ in rows[:10]:
        store.delete(key)
    live = store.live()
    print(first, store.compact())
    assert store.live() == live and len(live) == 90
    assert stable_id("seller-b", "0") != stable_id("seller-a", "0")
    print("Replay preserves identity; deletion hides rows before reclamation.")


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/21_operations/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None or "qdrant-secure" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra qdrant; start qdrant-secure and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Migrate real model collections, switch a native alias, roll back and restore.
    Run: uv run python modules/21_operations/examples/02_production.py
    CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
    Read the module's setup for optional extras and server requirements.
    """
    # course-extra: qdrant
    # course-profile: qdrant-secure

    from common.ops.native import native_migration_restore

    def main():
        result = native_migration_restore()
        assert result["restored_rows"] == 499
        assert result["rollback_count"] == 499
        print(result)

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/21_operations/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure native page reclamation and fresh-process disaster recovery.
Run: uv run python modules/21_operations/examples/03_measure.py
CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
Read the module's setup for optional extras and server requirements.
"""

from common.ops.drills import disaster_recovery, operations, sqlite_compaction


def main():
    print(operations())
    print(sqlite_compaction())
    recovered = disaster_recovery()
    assert recovered["lost_acknowledged_writes"] == 5
    assert recovered["resurrected_deletions"] == 0
    print(recovered)


if __name__ == "__main__":
    main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from tinyvec.storage import VersionedRows

s = VersionedRows()
s.upsert("a", 1)
s.upsert("a", 2)
print(s.live(), s.compact())

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.